In [59]:
import sys
import requests
import os
import pandas as pd
import io
import paramiko
import UnumEmail
from datetime import date

## Need to disconnect from the proxy to use this code

In [61]:
def exportSurvey(apiToken,surveyId, dataCenter, fileFormat):
    
    surveyId = surveyId
    fileFormat = fileFormat 
    dataCenter = dataCenter #organization ID

    # Setting static parameters
    requestCheckProgress = 0.0
    progressStatus = "inProgress"
    baseUrl = "https://{0}.az1.qualtrics.com/API/v3/surveys/{1}/export-responses/".format(dataCenter, surveyId)
    headers = {
    "content-type": "application/json",
    "x-api-token": apiToken,
    }

    # Step 1: Creating Data Export
    downloadRequestUrl = baseUrl
    downloadRequestPayload = '{"format":"' + fileFormat + '"}'
    downloadRequestResponse = requests.request("POST", downloadRequestUrl, data=downloadRequestPayload, headers=headers)
    progressId = downloadRequestResponse.json()["result"]["progressId"]
    print(downloadRequestResponse.text)

    # Step 2: Checking on Data Export Progress and waiting until export is ready
    while progressStatus != "complete" and progressStatus != "failed":
        print ("progressStatus=", progressStatus)
        requestCheckUrl = baseUrl + progressId
        requestCheckResponse = requests.request("GET", requestCheckUrl, headers=headers)
        requestCheckProgress = requestCheckResponse.json()["result"]["percentComplete"]
        print("Download is " + str(requestCheckProgress) + " complete")
        progressStatus = requestCheckResponse.json()["result"]["status"]

    #step 2.1: Check for error
    if progressStatus == "failed":
        raise Exception("export failed")

    fileId = requestCheckResponse.json()["result"]["fileId"]

    # Step 3: Downloading file
    requestDownloadUrl = baseUrl + fileId + '/file'
    requestDownload = requests.request("GET", requestDownloadUrl, headers=headers, stream=True)

    # Step 4: Unzipping the file
    zipfile.ZipFile(io.BytesIO(requestDownload.content)).extractall()
    print('Complete')

    
def main():
    
    apiToken = 'USv1f0D5J9XjC8N01x0yqGH3K9XiH17xTBIbwu3z'
    surveyId = 'SV_3dAVD7hE5Oz1sXj'
    dataCenter = 'unumhr'
    fileFormat = 'csv'

    if fileFormat not in ["csv", "tsv", "spss"]:
        print ('fileFormat must be either csv, tsv, or spss')
        sys.exit(2)
        
    r = re.compile('^SV_.*')
    m = r.match(surveyId)
    if not m:
        print ("survey Id must match ^SV_.*")
        sys.exit(2)

    exportSurvey(apiToken, surveyId,dataCenter, fileFormat)
    
if __name__ == "__main__":
    main()

{"result":{"progressId":"ES_aa7cT40lq64LxGZ","percentComplete":0.0,"status":"inProgress"},"meta":{"requestId":"d3658811-1f5c-4f2f-b65b-c4287b5fe233","httpStatus":"200 - OK"}}
progressStatus= inProgress
Download is 0.0 complete
progressStatus= inProgress
Download is 20.5761316872428 complete
progressStatus= inProgress
Download is 82.3045267489712 complete
progressStatus= inProgress
Download is 100.0 complete
Complete


In [6]:
df.columns.values.tolist()

['Start Date',
 'End Date',
 'Response Type',
 'IP Address',
 'Progress',
 'Duration (in seconds)',
 'Finished',
 'Recorded Date',
 'Response ID',
 'Location Latitude',
 'Location Longitude',
 'Thinking about your experience to date, how satisfied were you with the overall handling of your leave of absence or claim?\n\n\n0 = Extremely Unsatisfied, 10 = Extremely Satisfied',
 'Please elaborate on your answer below, any positive and/or constructive feedback you can share is helpful.',
 'Were you on a paid or unpaid leave of absence?',
 'Did you receive your pay and benefit deductions accurately and in a timely manner?',
 'Were your benefits handled as you expected them to be?',
 'Please describe either what went wrong or what did not go as expected with your pay and benefit deductions:',
 'For planning and filing your leave of absence or claim, did you primarily call in to the Absence Management Center (AMC) or use the Online Portal?',
 'How would you rate your interaction(s) with the Ab

In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv('C:\\Users\\ddy17\\Anaconda3\\Anaconda3\\envs\\p35env\\notebooks\\Employee Leave Experience Lifecycle.csv', header = 0,  skiprows=lambda x: x in [0, 2])
df_subset = df[['Response ID', 'Thinking about your experience to date, how satisfied were you with the overall handling of your leave of absence or claim?\n\n\n0 = Extremely Unsatisfied, 10 = Extremely Satisfied', 'Please elaborate on your answer below, any positive and/or constructive feedback you can share is helpful.','Were you on a paid or unpaid leave of absence?', 'Did you receive your pay and benefit deductions accurately and in a timely manner?', 'Were your benefits handled as you expected them to be?',
 'Please describe either what went wrong or what did not go as expected with your pay and benefit deductions:',
 'For planning and filing your leave of absence or claim, did you primarily call in to the Absence Management Center (AMC) or use the Online Portal?',
 'How would you rate your interaction(s) with the Absence Management Center?',
 'How would you rate your experience using the online portal?',
 'Are there any details you would like to share regarding your experience?',
 'Leading up to your return to work, who did you speak with? Please select all that apply:',
 'To the best of your knowledge, please rate your experience with the HR team(s) that you interacted with before, during, or after your return to work experience (if applicable): - HR Leave and Disability Consultant (formerly known as Health and Wellbeing Consultant)',
 'To the best of your knowledge, please rate your experience with the HR team(s) that you interacted with before, during, or after your return to work experience (if applicable): - HR Response Team or HR Leave Administration',
 'To the best of your knowledge, please rate your experience with the HR team(s) that you interacted with before, during, or after your return to work experience (if applicable): - Payroll',
 'Is there anything regarding your experience and interaction(s) with the previously selected team(s) or individual(s) that you would like to share?',
 'How was your experience when you returned to work?',
 'Please tell us about your overall return to work experience including the days leading up to your return and the time that followed:',
 'If you have any feedback, suggestions, or comments relating to your leave of absence or claim that were not addressed in this survey, please feel free to share with us below:',
 'The HR Leave Transformation Team holds focus groups from time to time to get our employees opinions on a variety of leave/absence related topics. If you would be interested in potentially participating in a focus group, please provide your name and email to be contacted for the next group.\n\n \n\nIf you would not like to participate, please leave the fields blank. Thank you! - Name',
 'The HR Leave Transformation Team holds focus groups from time to time to get our employees opinions on a variety of leave/absence related topics. If you would be interested in potentially participating in a focus group, please provide your name and email to be contacted for the next group.\n\n \n\nIf you would not like to participate, please leave the fields blank. Thank you! - Email',
 'Age Group','Business Area', 'Campus', 'Country', 'Department', 'Return To Work Date', 'Survey Launch Date', 'Leave Launch Date']].copy()

df_subset

,Response ID,"Thinking about your experience to date, how satisfied were you with the overall handling of your leave of absence or claim?\n\n\n0 = Extremely Unsatisfied, 10 = Extremely Satisfied","Please elaborate on your answer below, any positive and/or constructive feedback you can share is helpful.",Were you on a paid or unpaid leave of absence?,Did you receive your pay and benefit deductions accurately and in a timely manner?,Were your benefits handled as you expected them to be?,Please describe either what went wrong or what did not go as expected with your pay and benefit deductions:,"For planning and filing your leave of absence or claim, did you primarily call in to the Absence Management Center (AMC) or use the Online Portal?",How would you rate your interaction(s) with the Absence Management Center?,How would you rate your experience using the online portal?,Are there any details you would like to share regarding your experience?,"Leading up to your return to work, who did you speak with? Please select all that apply:","To the best of your knowledge, please rate your experience with the HR team(s) that you interacted with before, during, or after your return to work experience (if applicable): - HR Leave and Disability Consultant (formerly known as Health and Wellbeing Consultant)","To the best of your knowledge, please rate your experience with the HR team(s) that you interacted with before, during, or after your return to work experience (if applicable): - HR Response Team or HR Leave Administration","To the best of your knowledge, please rate your experience with the HR team(s) that you interacted with before, during, or after your return to work experience (if applicable): - Payroll",Is there anything regarding your experience and interaction(s) with the previously selected team(s) or individual(s) that you would like to share?,How was your experience when you returned to work?,Please tell us about your overall return to work experience including the days leading up to your return and the time that followed:,"If you have any feedback, suggestions, or comments relating to your leave of absence or claim that were not addressed in this survey, please feel free to share with us below:","The HR Leave Transformation Team holds focus groups from time to time to get our employees opinions on a variety of leave/absence related topics. If you would be interested in potentially participating in a focus group, please provide your name and email to be contacted for the next group.\n\n \n\nIf you would not like to participate, please leave the fields blank. Thank you! - Name","The HR Leave Transformation Team holds focus groups from time to time to get our employees opinions on a variety of leave/absence related topics. If you would be interested in potentially participating in a focus group, please provide your name and email to be contacted for the next group.\n\n \n\nIf you would not like to participate, please leave the fields blank. Thank you! - Email",Age Group,Business Area,Campus,Country,Department,Return To Work Date,Survey Launch Date,Leave Launch Date
0,R_4Idj0xT0s0tYkE5,5.0,NaN,1.0,12.0,NaN,NaN,0.0,2.0,NaN,NaN,"25,26",NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,30 - 34,Unum US Direct,Portland,United States of America,Unum US Benefits Operations,4/2/2020,6/22/2020,NaN
1,R_eECFLVLCaD9YTEp,8.0,NaN,1.0,12.0,NaN,Even though STD disability was submitted withi...,0.0,4.0,NaN,NaN,"25,26,27",4.0,4.0,3.0,NaN,4.0,NaN,NaN,NaN,NaN,45 - 49,Unum US Direct,Portland,United States of America,Unum US Benefits Operations,4/22/2020,6/22/2020,NaN
2,R_cXUamAyLEXmwogR,7.0,NaN,1.0,12.0,NaN,the way you're supposed to enter your time out...,1.0,NaN,3.0,NaN,"25,26",NaN,NaN,NaN,NaN,3.0,had technical difficulties with my cpu both ti...,have a different way for employees to enter th...,Drew Roden,droden@unum.com,30 - 34,Unum US Direct,Chattanooga,United States of America,Unum US Benefits AMC and CEC,4/6/2020,6/22/2020,NaN
3,R_1H4pSqyCCglm96Z,8.0,NaN,1.

In [ ]:
recodes = {"favorable":     {"favorable": 4, "two": 2},
                "num_cylinders": {"four": 4, "six": 6, "five": 5, "eight": 8,
                                  "two": 2, "twelve": 12, "three":3 }}

In [ ]:
date = (date.today()).strftime('%m/%d')

In [ ]:
sender = 'do_not_reply@chelsea.com'
recipients = ['cwymer@unum.com']
subject = 'Leave Experience Data Export' 
body = (date,'Leave Data Export from Qualtrics is attached to this email')

In [ ]:
UnumEmail.send_email(sender, recipients, subject, body, file_attached = True, attachment = 'qualtrics_api.ipynb', attach_path = './qualtrics_api.ipynb')

## Move .csv export from 'notebooks' to 'csv_exports_api' for clean python folders

In [ ]:
os.chdir('C:\\Users\\ddy17\\Anaconda3\\ENVS\\p35workshop\\csv_exports_api')